In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import os

from pathlib import Path

In [5]:
def read_realsense_bag(bag_file_path, output_path):
    # Configure the pipeline
    pipeline = rs.pipeline()
    config = rs.config()

    # Enable playback of the bag file
    config.enable_device_from_file(bag_file_path, repeat_playback=False)

    # Start the pipeline
    pipeline.start(config)

    os.makedirs(output_path, exist_ok=True)
    os.makedirs(output_path / 'rgb', exist_ok=True)
    os.makedirs(output_path / 'depth', exist_ok=True)

    i = 0

    try:
        while True:
            # Wait for a frame from the pipeline
            frames = pipeline.wait_for_frames()

            # Get color and depth frames
            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()

            if not color_frame or not depth_frame:
                break

            # Convert images to numpy arrays
            color_image = cv2.cvtColor(np.asanyarray(color_frame.get_data()), cv2.COLOR_BGR2RGB)
            depth_image = np.asanyarray(depth_frame.get_data())

            num = str(i)
            while len(num) < 4:
                num = '0' + num

            # Display the images
            cv2.imwrite(str(output_path / f'rgb/{num}.png'), color_image)
            cv2.imwrite(str(output_path / f'depth/{num}.png'), depth_image)

            i += 1
    finally:
        # Stop the pipeline
        pipeline.stop()

In [7]:
bag_file_path = '/media/scratch/Datasets/zie029/datasets/realsense_data/realsense-after.bag'  # Replace with your bag file path
output_path = Path('/workspace/Neo/data/realsense_after')
read_realsense_bag(bag_file_path, output_path)

RuntimeError: Frame didn't arrive within 5000